# Strict Partial Top-10 Human Review Export

This notebook creates human-review tables from the already-generated RefMet evaluation outputs. It does not run embeddings, FAISS, or reranking again.

The review set starts from the conservative evaluation and then removes query/gold pairs that appear to be surface-trivial variants: exact normalized matches, punctuation-only compact matches, substring variants, high sequence-similarity variants, or token-subset variants. Candidate names that are surface-trivial variants of the query are also removed before forming the top-10 list.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

CANDIDATE_PROJECT_DIRS = [
    Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
    Path('/content/drive/MyDrive/ChemSynonymizer'),
    Path('/content/chem_synonymizer'),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_DIR = next((p for p in CANDIDATE_PROJECT_DIRS if (p / 'results').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find project directory. Set PROJECT_DIR manually in this cell.')

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('Working directory:', Path.cwd())

## Run Export

Adjust the thresholds below if needed. A lower sequence-similarity threshold removes more query/gold pairs as surface-trivial. A higher threshold keeps more examples.

In [ ]:
import runpy
import sys

TOP_K = 10
SURFACE_SIM_THRESHOLD = 0.90
SUBSTRING_MIN_LEN = 5
SUBSTRING_MIN_LEN_RATIO = 0.50

script_path = PROJECT_DIR / 'scripts' / 'make_strict_partial_top10_human_review.py'
if not script_path.exists():
    raise FileNotFoundError(f'Missing script: {script_path}')

sys.argv = [
    str(script_path),
    '--project-dir', str(PROJECT_DIR),
    '--top-k', str(TOP_K),
    '--surface-sim-threshold', str(SURFACE_SIM_THRESHOLD),
    '--substring-min-len', str(SUBSTRING_MIN_LEN),
    '--substring-min-len-ratio', str(SUBSTRING_MIN_LEN_RATIO),
]
runpy.run_path(str(script_path), run_name='__main__')

## Review Generated Files

In [ ]:
import pandas as pd

OUT_DIR = PROJECT_DIR / 'results' / 'publication_readiness'

summary = pd.read_csv(OUT_DIR / 'strict_partial_top10_review_summary.csv')
metrics = pd.read_csv(OUT_DIR / 'strict_partial_top10_review_metrics.csv')
relation_counts = pd.read_csv(OUT_DIR / 'strict_partial_query_gold_surface_relation_counts.csv')

display(summary)
display(metrics)
display(relation_counts)

In [ ]:
sample_wide = pd.read_csv(OUT_DIR / 'strict_partial_top10_human_review_sample_wide.csv')
sample_long = pd.read_csv(OUT_DIR / 'strict_partial_top10_human_review_sample_long.csv')

print('Sample wide:', sample_wide.shape)
print('Sample long:', sample_long.shape)
display(sample_wide.head(10))
display(sample_long.head(20))

## Output Files

The main review files are saved in `results/publication_readiness/`:

- `strict_partial_top10_human_review_wide.csv`: one row per query, with reranker-ranked top-10 candidates in columns.
- `strict_partial_top10_human_review_long.csv`: one row per query-candidate pair, easier for annotation/filtering.
- `strict_partial_top10_human_review_sample_wide.csv`: stratified sample for practical manual review.
- `strict_partial_top10_human_review_sample_long.csv`: long-format version of the stratified sample.
- `strict_partial_top10_human_review_reranker_wide.csv`: explicit reranker-named copy of the wide review table.
- `strict_partial_top10_human_review_reranker_long.csv`: explicit reranker-named copy of the long review table.
- `strict_partial_top10_human_review_reranker_sample_wide.csv`: explicit reranker-named stratified sample.
- `strict_partial_top10_human_review_reranker_sample_long.csv`: explicit reranker-named long sample.
- `strict_partial_top10_review_metrics.csv`: metrics after the stricter query/gold and candidate/query surface filtering.
- `strict_partial_top10_review_summary.csv`: count summary explaining how many rows were removed and why.